In [1]:
from pyspark.sql.functions import (
    col, when, trim, upper, lower, regexp_replace,
    to_timestamp, to_date, current_timestamp, lit,
    input_file_name, count, sum as spark_sum,
    row_number, sha2, concat_ws, round as spark_round
)
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, IntegerType
from datetime import datetime

BRONZE_TABLE    = "Interac_Bronze.dbo.transactions"
SILVER_TABLE    = "silver_transactions"
SILVER_DB       = "Interac_Fabric_Workspace.Interac_Silver.dbo"
PIPELINE_NAME   = "NB_02_Silver_Transactions"
BATCH_DATE      = datetime.now().strftime("%Y-%m-%d")

print(f"Silver Transactions Pipeline")
print(f"Source : {BRONZE_TABLE}")
print(f"Target : {SILVER_DB}.{SILVER_TABLE}")
print(f"Started: {datetime.now()}")

StatementMeta(, 5a426d5d-2a7b-45a1-aa62-ac4e5387f2c1, 3, Finished, Available, Finished, False)

Silver Transactions Pipeline
Source : Interac_Bronze.dbo.transactions
Target : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_transactions
Started: 2026-05-06 00:17:11.159772


In [2]:
# ── Read from Bronze ────────────────────────────────────────
df_bronze = spark.read.table(BRONZE_TABLE)

total_bronze = df_bronze.count()
print(f"Bronze rows read: {total_bronze:,}")

# ── Data Quality Checks ─────────────────────────────────────
print("\nRunning DQ checks...")

dq_results = {}

null_txn_id = df_bronze.filter(col("transaction_id").isNull()).count()
dq_results["null_transaction_id"] = null_txn_id

null_ch = df_bronze.filter(col("cardholder_id").isNull()).count()
dq_results["null_cardholder_id"] = null_ch

null_mer = df_bronze.filter(col("merchant_id").isNull()).count()
dq_results["null_merchant_id"] = null_mer

invalid_amount = df_bronze.filter(
    col("amount_cad").isNull() | (col("amount_cad") <= 0)
).count()
dq_results["invalid_amount"] = invalid_amount

valid_codes = ["00","51","54","57","61","62","91","96"]
invalid_codes = df_bronze.filter(
    ~col("response_code").isin(valid_codes)
).count()
dq_results["invalid_response_code"] = invalid_codes

international = df_bronze.filter(col("is_international") == "Y").count()
dq_results["international_transactions"] = international

suspicious = df_bronze.filter(col("is_flagged") == "Y").count()
dq_results["suspicious_transactions"] = suspicious

print("\nDQ CHECK RESULTS:")
print("-" * 45)
for check, count_val in dq_results.items():
    status = "⚠ FLAGGED" if count_val > 0 else "✓ PASSED"
    print(f"{check:<35} {count_val:>6,}  {status}")

StatementMeta(, 5a426d5d-2a7b-45a1-aa62-ac4e5387f2c1, 4, Finished, Available, Finished, False)

Bronze rows read: 145,946

Running DQ checks...

DQ CHECK RESULTS:
---------------------------------------------
null_transaction_id                      0  ✓ PASSED
null_cardholder_id                       0  ✓ PASSED
null_merchant_id                         0  ✓ PASSED
invalid_amount                           0  ✓ PASSED
invalid_response_code               77,953  ⚠ FLAGGED
international_transactions           4,412  ⚠ FLAGGED
suspicious_transactions              1,548  ⚠ FLAGGED


In [3]:
# ── Quarantine bad records ──────────────────────────────────
df_quarantine = df_bronze.filter(
    col("transaction_id").isNull() |
    col("cardholder_id").isNull() |
    col("merchant_id").isNull() |
    col("amount_cad").isNull() |
    (col("amount_cad") <= 0)
)

quarantine_count = df_quarantine.count()

if quarantine_count > 0:
    (df_quarantine
        .withColumn("_quarantine_reason", lit("FAILED_HARD_DQ_RULES"))
        .withColumn("_quarantined_at", current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_DB}.silver_transactions_quarantine"))
    print(f"Quarantined: {quarantine_count:,} records")

# ── Keep only valid records ─────────────────────────────────
df_valid = df_bronze.filter(
    col("transaction_id").isNotNull() &
    col("cardholder_id").isNotNull() &
    col("merchant_id").isNotNull() &
    col("amount_cad").isNotNull() &
    (col("amount_cad") > 0)
)

print(f"Valid records: {df_valid.count():,}")

StatementMeta(, 5a426d5d-2a7b-45a1-aa62-ac4e5387f2c1, 5, Finished, Available, Finished, False)

Valid records: 145,946


In [4]:
# ── Apply Silver transformations ────────────────────────────
df_silver = (df_valid
    .withColumn("transaction_id",   trim(col("transaction_id")))
    .withColumn("cardholder_id",    trim(col("cardholder_id")))
    .withColumn("merchant_id",      trim(col("merchant_id")))
    .withColumn("transaction_type", upper(trim(col("transaction_type"))))
    .withColumn("payment_method",   upper(trim(col("payment_method"))))
    .withColumn("channel",          upper(trim(col("channel"))))
    .withColumn("network",          upper(trim(col("network"))))
    .withColumn("currency",         upper(trim(col("currency"))))
    .withColumn("is_approved",      upper(trim(col("is_approved"))))
    .withColumn("is_international", upper(trim(col("is_international"))))
    .withColumn("is_flagged",       upper(trim(col("is_flagged"))))
    .withColumn("amount_cad",
        col("amount_cad").cast(DecimalType(18, 2)))
    .withColumn("interchange_fee_cad",
        col("interchange_fee_cad").cast(DecimalType(18, 4)))
    .withColumn("processing_time_ms",
        col("processing_time_ms").cast(IntegerType()))
    .withColumn("transaction_datetime",
        to_timestamp(col("transaction_datetime"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("transaction_date",
        to_date(col("transaction_date"), "yyyy-MM-dd"))
    .withColumn("transaction_hour",
        col("transaction_datetime").cast("string").substr(12, 2).cast(IntegerType()))
    .withColumn("transaction_month",
        col("transaction_datetime").cast("string").substr(1, 7))
    .withColumn("amount_band",
        when(col("amount_cad") < 25,   "MICRO")
        .when(col("amount_cad") < 100,  "SMALL")
        .when(col("amount_cad") < 500,  "MEDIUM")
        .when(col("amount_cad") < 2000, "LARGE")
        .otherwise("VERY_LARGE"))
    .withColumn("authorization_code",
        when(col("authorization_code") == "", None)
        .otherwise(col("authorization_code")))
    .withColumn("_silver_loaded_at",  current_timestamp())
    .withColumn("_pipeline_name",     lit(PIPELINE_NAME))
    .withColumn("_batch_date",        lit(BATCH_DATE))
    .drop("_ingested_at", "_source_file", "_lakehouse")
)

print(f"Transformed rows: {df_silver.count():,}")

StatementMeta(, 5a426d5d-2a7b-45a1-aa62-ac4e5387f2c1, 6, Finished, Available, Finished, False)

Transformed rows: 145,946


In [5]:
# ── Write to Silver ─────────────────────────────────────────
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}"))

spark.sql(f"OPTIMIZE {SILVER_DB}.{SILVER_TABLE} ZORDER BY (transaction_date, cardholder_id)")

final_count = spark.read.table(f"{SILVER_DB}.{SILVER_TABLE}").count()

print("\n" + "="*60)
print("SILVER TRANSACTIONS SUMMARY")
print("="*60)
print(f"Bronze rows in      : {total_bronze:,}")
print(f"Quarantined         : {quarantine_count:,}")
print(f"Silver rows out     : {final_count:,}")
print(f"Pass rate           : {round(final_count/total_bronze*100, 2)}%")
print(f"Table               : {SILVER_DB}.{SILVER_TABLE}")
print(f"Completed at        : {datetime.now()}")
print("="*60)

StatementMeta(, 5a426d5d-2a7b-45a1-aa62-ac4e5387f2c1, 7, Finished, Available, Finished, False)


SILVER TRANSACTIONS SUMMARY
Bronze rows in      : 145,946
Quarantined         : 0
Silver rows out     : 145,946
Pass rate           : 100.0%
Table               : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_transactions
Completed at        : 2026-05-06 00:18:13.284880
